# 7. Support Vector Machines

**Machine Learning Fundamentals and Predictive Analytics — Notebook 7 of 11**

Logistic regression finds *a* line that separates the classes. A **support vector machine**
finds the **best** one: the boundary with the largest possible **margin** — the widest empty
corridor between the two classes.

Then it does something cleverer. Via the **kernel trick**, an SVM can find a linear boundary in
a vastly higher-dimensional space *without ever computing the coordinates in that space*. That
gives it non-linear power at linear cost, and it made SVMs the dominant classifier of the
1990s and 2000s.

### What you will learn

1. The **maximum margin** idea, and what a **support vector** is
2. **Hard vs soft margin**, and the `C` parameter
3. **Hinge loss** — how the optimisation is actually posed
4. The **kernel trick**: linear, polynomial, RBF, sigmoid
5. **`gamma`** and what it controls
6. Why **scaling is mandatory** for SVMs
7. Tuning `C` and `gamma` together
8. **Multiclass** SVMs, probabilities, and calibration
9. **SVR** — support vector regression
10. Strengths, weaknesses and computational limits

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC, SVR, LinearSVC, LinearSVR
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     StratifiedKFold, KFold, validation_curve)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix, mean_squared_error, r2_score,
                             average_precision_score, recall_score, precision_score,
                             log_loss)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import make_blobs, make_moons, make_circles, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import time

rng = np.random.default_rng(seed=7)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)
CV = KFold(5, shuffle=True, random_state=0)

---
## 7.1 The maximum margin

For linearly separable data there are infinitely many separating lines. Which is best?

The SVM answer: the one **furthest from the nearest point of either class**. Formally, with the
decision function $f(\mathbf{x}) = \mathbf{w}^\top\mathbf{x} + b$ and labels $y_i \in \{-1, +1\}$:

$$\max_{\mathbf{w}, b}\ \frac{2}{\|\mathbf{w}\|} \quad\text{subject to}\quad
y_i(\mathbf{w}^\top\mathbf{x}_i + b) \ge 1 \ \ \forall i$$

equivalently

$$\min_{\mathbf{w}, b}\ \tfrac{1}{2}\|\mathbf{w}\|^2 \quad\text{subject to the same constraints}$$

The **margin** is the corridor of width $2/\|\mathbf{w}\|$ between the two dashed lines
$f(\mathbf{x}) = \pm 1$.

**Support vectors** are the points lying exactly on those dashed lines (or, with a soft margin,
inside the corridor). They are the only points that matter: delete every other training point
and you get the identical model. That is a remarkable property — the model is defined by a
handful of examples.

Why maximise the margin? A wide margin means the boundary is far from the data, so small
perturbations do not change the prediction. It is a form of regularisation, and it comes with
generalisation guarantees.

In [ ]:
X, y = make_blobs(n_samples=60, centers=2, cluster_std=1.05, random_state=6)

svm = SVC(kernel="linear", C=1e6).fit(X, y)          # huge C = essentially a hard margin
w, b = svm.coef_[0], svm.intercept_[0]

def plot_svm_margin(model, X, y, ax, title):
    xx = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 300)
    yy = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 300)
    XX, YY = np.meshgrid(xx, yy)
    Z = model.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contourf(XX, YY, Z > 0, alpha=0.12, cmap="coolwarm")
    ax.contour(XX, YY, Z, levels=[-1, 0, 1], colors="k",
               linestyles=["--", "-", "--"], linewidths=[1, 2, 1])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=40, edgecolor="k", linewidth=0.4)
    sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=180, facecolors="none", edgecolors="gold", linewidths=2.2,
               label=f"{len(sv)} support vectors")
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8)

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_svm_margin(svm, X, y, ax, "Maximum-margin classifier")
plt.show()

print(f"Weight vector w = {np.round(w, 4)}, intercept b = {b:.4f}")
print(f"Margin width = 2/||w|| = {2/np.linalg.norm(w):.4f}")
print(f"Support vectors: {len(svm.support_vectors_)} out of {len(X)} training points")
print(f"Their indices  : {svm.support_.tolist()}")

In [ ]:
# Only the support vectors matter -- delete everything else and refit
keep = svm.support_
svm_sv_only = SVC(kernel="linear", C=1e6).fit(X[keep], y[keep])

print(f"Full data model : w = {np.round(svm.coef_[0], 6)}, b = {svm.intercept_[0]:.6f}")
print(f"SVs-only model  : w = {np.round(svm_sv_only.coef_[0], 6)}, "
      f"b = {svm_sv_only.intercept_[0]:.6f}")
print(f"\nIdentical decision function? "
      f"{np.allclose(svm.coef_, svm_sv_only.coef_) and np.allclose(svm.intercept_, svm_sv_only.intercept_)}")
print(f"Predictions agree on all points? {(svm.predict(X) == svm_sv_only.predict(X)).all()}")
print(f"\nWe threw away {len(X) - len(keep)} of {len(X)} training points and lost nothing.")
print("Contrast with logistic regression, where every point contributes to the loss.")

# Compare with logistic regression on the same data
logit = make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000)).fit(X, y)
fig, ax = plt.subplots(figsize=(7, 5))
xx = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 300)
plot_svm_margin(svm, X, y, ax, "SVM boundary (solid) vs logistic regression (green)")
w2 = logit[-1].coef_[0] / StandardScaler().fit(X).scale_
b2 = logit[-1].intercept_[0] - (logit[-1].coef_[0] * StandardScaler().fit(X).mean_
                                / StandardScaler().fit(X).scale_).sum()
ax.plot(xx, -(w2[0]*xx + b2)/w2[1], color="seagreen", lw=2.4, label="logistic regression")
ax.legend(fontsize=8)
plt.show()
print("Both separate the data. The SVM deliberately centres its boundary in the gap;")
print("logistic regression is pulled by every point, including those far from the boundary.")

---
## 7.2 Soft margin and the `C` parameter

Real data is not separable, and even when it is, a hard margin chases outliers. The **soft
margin** allows violations, paying a penalty:

$$\min_{\mathbf{w}, b, \xi}\ \tfrac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i}\xi_i
\quad\text{s.t.}\quad y_i(\mathbf{w}^\top\mathbf{x}_i + b) \ge 1 - \xi_i,\ \ \xi_i \ge 0$$

where $\xi_i$ is the **slack** for point $i$ — how far it intrudes into (or past) the margin.

**`C` is the cost of a violation:**

| `C` | Behaviour |
|---|---|
| **small** (0.01) | Violations are cheap → **wide** margin, many support vectors, **more bias, less variance** |
| **large** (1000) | Violations are expensive → **narrow** margin, few support vectors, **less bias, more variance** |

So `C` is an inverse regularisation parameter, exactly like `C` in `LogisticRegression`. Small
`C` = strong regularisation.

In [ ]:
X_ov, y_ov = make_blobs(n_samples=100, centers=2, cluster_std=1.9, random_state=4)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, C in zip(axes, [0.01, 0.1, 1.0, 100.0]):
    m = SVC(kernel="linear", C=C).fit(X_ov, y_ov)
    plot_svm_margin(m, X_ov, y_ov, ax,
                    f"C = {C}\nmargin width {2/np.linalg.norm(m.coef_[0]):.3f}, "
                    f"{len(m.support_)} SVs")
plt.tight_layout(); plt.show()

print(f"{'C':>10}{'margin width':>15}{'support vectors':>17}{'train acc':>11}{'CV acc':>9}")
for C in (0.001, 0.01, 0.1, 1, 10, 100, 1000):
    m = SVC(kernel="linear", C=C).fit(X_ov, y_ov)
    cvv = cross_val_score(SVC(kernel="linear", C=C), X_ov, y_ov, cv=SKF).mean()
    print(f"{C:>10}{2/np.linalg.norm(m.coef_[0]):>15.4f}{len(m.support_):>17}"
          f"{m.score(X_ov, y_ov):>11.4f}{cvv:>9.4f}")
print("\nSmall C: a wide margin that ignores individual points -- many of them end up inside")
print("the corridor and become support vectors. Large C: the boundary contorts to satisfy")
print("every point it can.")

### Hinge loss

The soft-margin problem is equivalent to minimising **hinge loss** plus an $L_2$ penalty:

$$\min_{\mathbf{w},b}\ \underbrace{\frac{1}{n}\sum_{i}\max\big(0,\ 1 - y_i f(\mathbf{x}_i)\big)}_{\text{hinge loss}}
\ +\ \lambda\|\mathbf{w}\|^2, \qquad \lambda \propto \frac{1}{C}$$

Compare it with the log loss of logistic regression:

- **Hinge loss is exactly zero** once a point is correctly classified with margin $\ge 1$.
  Those points contribute nothing — which is why only the support vectors matter.
- **Log loss is never zero.** Every point keeps pushing, however confidently classified.

That single difference explains most of the behavioural gap between the two models.

In [ ]:
margin = np.linspace(-2, 3, 500)          # y * f(x)
hinge = np.maximum(0, 1 - margin)
logl = np.log(1 + np.exp(-margin))
zero_one = (margin < 0).astype(float)

plt.plot(margin, hinge, lw=2.4, color="steelblue", label="hinge loss (SVM)")
plt.plot(margin, logl, lw=2.4, color="crimson", label="log loss (logistic regression)")
plt.plot(margin, zero_one, lw=1.8, ls="--", color="grey", label="0-1 loss (what we want)")
plt.axvline(1, color="black", ls=":", lw=1)
plt.text(1.05, 2.2, "margin = 1", fontsize=8)
plt.xlabel("y * f(x)  (positive = correctly classified)"); plt.ylabel("loss")
plt.ylim(-0.1, 3); plt.title("Hinge loss switches off; log loss never does")
plt.legend(fontsize=8); plt.show()

print("Loss at a few margins:")
print(f"{'y*f(x)':>9}{'hinge':>9}{'log loss':>11}")
for mg in (-1, 0, 0.5, 1.0, 2.0, 5.0):
    print(f"{mg:>9.1f}{max(0, 1-mg):>9.3f}{np.log(1+np.exp(-mg)):>11.3f}")
print("\nBoth losses are convex upper bounds on the 0-1 loss, which is what makes them")
print("tractable substitutes for 'minimise the number of mistakes'.")

---
## 7.3 The kernel trick

Some data is not linearly separable in its original space but becomes separable after a
transformation $\phi(\mathbf{x})$ into a higher-dimensional space.

The insight: the SVM optimisation only ever needs **inner products** between points. So if we
have a function

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^\top\phi(\mathbf{x}_j)$$

we can work in the transformed space **without ever computing $\phi$**. That is the **kernel
trick**, and it means the feature space can even be infinite-dimensional.

| Kernel | $K(\mathbf{x}, \mathbf{x}')$ | Parameters | Use for |
|---|---|---|---|
| **linear** | $\mathbf{x}^\top\mathbf{x}'$ | — | High-dimensional, text, $p > n$ |
| **polynomial** | $(\gamma\,\mathbf{x}^\top\mathbf{x}' + r)^d$ | `degree`, `gamma`, `coef0` | Known polynomial interactions |
| **RBF (Gaussian)** | $\exp(-\gamma\|\mathbf{x}-\mathbf{x}'\|^2)$ | `gamma` | **The default choice** — smooth, local, very flexible |
| **sigmoid** | $\tanh(\gamma\,\mathbf{x}^\top\mathbf{x}' + r)$ | `gamma`, `coef0` | Rarely; not always a valid kernel |

Start with RBF. Use linear when $p$ is large or you need speed and interpretability.

In [ ]:
# Why a transformation helps: a 1-D example made separable by adding x^2
x1 = np.r_[rng.uniform(-3, -1, 25), rng.uniform(-0.8, 0.8, 30), rng.uniform(1, 3, 25)]
y1 = np.r_[np.zeros(25), np.ones(30), np.zeros(25)].astype(int)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(x1, np.zeros_like(x1), c=y1, cmap="coolwarm", s=60, edgecolor="k", linewidth=0.4)
ax[0].set_yticks([]); ax[0].set_xlabel("x")
ax[0].set_title("In 1-D: no single threshold separates the classes")
ax[1].scatter(x1, x1**2, c=y1, cmap="coolwarm", s=60, edgecolor="k", linewidth=0.4)
ax[1].axhline(1.0, color="black", lw=2, ls="--")
ax[1].set_xlabel("x"); ax[1].set_ylabel("x squared")
ax[1].set_title("Add x^2: a horizontal line separates them perfectly")
plt.tight_layout(); plt.show()

print(f"Linear SVM on x alone       : accuracy "
      f"{SVC(kernel='linear').fit(x1.reshape(-1,1), y1).score(x1.reshape(-1,1), y1):.4f}")
print(f"Linear SVM on [x, x^2]      : accuracy "
      f"{SVC(kernel='linear').fit(np.c_[x1, x1**2], y1).score(np.c_[x1, x1**2], y1):.4f}")
print(f"RBF SVM on x alone          : accuracy "
      f"{SVC(kernel='rbf').fit(x1.reshape(-1,1), y1).score(x1.reshape(-1,1), y1):.4f}")
print("\nThe RBF kernel reaches the same place without our having to think of x^2.")

In [ ]:
# The four kernels on three dataset shapes
def boundary(model, X, y, ax, title):
    h = 0.02
    xx, yy = np.meshgrid(np.arange(X[:, 0].min()-0.6, X[:, 0].max()+0.6, h),
                         np.arange(X[:, 1].min()-0.6, X[:, 1].max()+0.6, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=16, edgecolor="k", linewidth=0.2)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

shapes = {
    "linearly separable": make_blobs(n_samples=300, centers=2, cluster_std=1.4, random_state=3),
    "moons": make_moons(n_samples=300, noise=0.18, random_state=3),
    "circles": make_circles(n_samples=300, noise=0.10, factor=0.45, random_state=3),
}
kernels = ["linear", "poly", "rbf", "sigmoid"]

fig, axes = plt.subplots(len(shapes), len(kernels), figsize=(16, 11))
for r, (sname, (Xs, ys)) in enumerate(shapes.items()):
    Xs = StandardScaler().fit_transform(Xs)
    for c, kern in enumerate(kernels):
        mdl = SVC(kernel=kern, C=1.0, degree=3, gamma="scale").fit(Xs, ys)
        acc = cross_val_score(SVC(kernel=kern, C=1.0, gamma="scale"), Xs, ys, cv=SKF).mean()
        boundary(mdl, Xs, ys, axes[r, c],
                 f"{kern if r == 0 else ''}\n{sname}: CV acc {acc:.3f}")
plt.tight_layout(); plt.show()

print("The RBF kernel handles all three shapes. The linear kernel handles only the first.")
print("The polynomial kernel manages the circles (a quadratic boundary) but is awkward on")
print("the moons. The sigmoid kernel is erratic -- there is a reason nobody uses it.")

---
## 7.4 `gamma`: how far a single point's influence reaches

For the RBF kernel $K(\mathbf{x},\mathbf{x}') = \exp(-\gamma\|\mathbf{x}-\mathbf{x}'\|^2)$,
$\gamma$ sets the width of the bump around each training point:

| `gamma` | Influence radius | Effect |
|---|---|---|
| **small** | wide | Smooth, almost linear boundary → **high bias** |
| **large** | narrow | Boundary wraps tightly around individual points → **high variance** |

Very large `gamma` gives you an expensive nearest-neighbour classifier: each training point
influences only itself.

`gamma="scale"` (the default) uses $\gamma = \frac{1}{p\cdot\operatorname{Var}(X)}$, which
adapts to the data — one of the reasons scaling matters so much.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.22, random_state=1)
Xm = StandardScaler().fit_transform(Xm)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0,
                                              stratify=ym)

fig, axes = plt.subplots(1, 5, figsize=(19, 4))
for ax, g in zip(axes, [0.01, 0.1, 1.0, 10.0, 200.0]):
    mdl = SVC(kernel="rbf", C=1.0, gamma=g).fit(Xm_tr, ym_tr)
    boundary(mdl, Xm_tr, ym_tr, ax,
             f"gamma = {g}\ntrain {mdl.score(Xm_tr, ym_tr):.3f} / "
             f"test {mdl.score(Xm_te, ym_te):.3f}\n{len(mdl.support_)} SVs")
plt.tight_layout(); plt.show()

print(f"{'gamma':>9}{'train acc':>11}{'test acc':>10}{'SVs':>7}")
for g in (0.001, 0.01, 0.1, 1, 10, 100, 1000):
    mdl = SVC(kernel="rbf", C=1.0, gamma=g).fit(Xm_tr, ym_tr)
    print(f"{g:>9}{mdl.score(Xm_tr, ym_tr):>11.4f}{mdl.score(Xm_te, ym_te):>10.4f}"
          f"{len(mdl.support_):>7}")
print("\nAt gamma=1000 the model memorises the training set (accuracy 1.0) and nearly")
print("every point becomes a support vector -- the classic signature of overfitting.")
print(f"\ngamma='scale' would use {1/(Xm_tr.shape[1] * Xm_tr.var()):.4f} here.")

---
## 7.5 Scaling is mandatory

Both the RBF kernel ($\|\mathbf{x}-\mathbf{x}'\|^2$) and the margin ($\|\mathbf{w}\|$) depend
on the scale of the features. An unscaled SVM:

- lets the largest-range feature dominate the kernel, exactly as with KNN
- converges slowly or not at all
- makes `gamma="scale"` and any `C` you chose meaningless

**Always wrap an SVM in a pipeline with a scaler.** This is not a nice-to-have.

In [ ]:
# Two informative features, wildly different scales
m = 800
f_big = rng.normal(500_000, 120_000, m)
f_small = rng.normal(3.0, 0.9, m)
z = (1.1*(f_big - f_big.mean())/f_big.std() + 1.1*(f_small - f_small.mean())/f_small.std())
lab = (z + rng.normal(0, 0.6, m) > 0).astype(int)
Xu = np.column_stack([f_big, f_small])
Xa, Xb, ya, yb = train_test_split(Xu, lab, test_size=0.3, random_state=0, stratify=lab)

t0 = time.perf_counter()
raw = SVC(kernel="rbf").fit(Xa, ya)
t_raw = time.perf_counter() - t0
t0 = time.perf_counter()
scaled = make_pipeline(StandardScaler(), SVC(kernel="rbf")).fit(Xa, ya)
t_scaled = time.perf_counter() - t0

print(f"{'model':<24}{'test acc':>10}{'SVs':>8}{'fit time (s)':>14}")
print(f"{'unscaled':<24}{raw.score(Xb, yb):>10.4f}{len(raw.support_):>8}{t_raw:>14.3f}")
print(f"{'scaled':<24}{scaled.score(Xb, yb):>10.4f}"
      f"{len(scaled[-1].support_):>8}{t_scaled:>14.3f}")
print(f"{'baseline':<24}"
      f"{DummyClassifier(strategy='most_frequent').fit(Xa, ya).score(Xb, yb):>10.4f}")
print("\nUnscaled, nearly every training point becomes a support vector -- the kernel")
print("cannot distinguish anything, because all distances are dominated by f_big.")
print("The model is both slower and worse.")

print("\nWhich scaler? Both work; StandardScaler is the usual choice.")
for name, sc in [("StandardScaler", StandardScaler()), ("MinMaxScaler", MinMaxScaler())]:
    print(f"  {name:<16} CV accuracy "
          f"{cross_val_score(make_pipeline(sc, SVC()), Xu, lab, cv=SKF).mean():.4f}")

---
## 7.6 Tuning `C` and `gamma` together

`C` and `gamma` **interact**, so they must be tuned jointly, not one after the other. The
standard approach is a grid over powers of 10, then a finer grid around the winner.

- High `C` + high `gamma` → severe overfitting
- Low `C` + low `gamma` → underfitting
- The good region is usually a diagonal band

In [ ]:
Cs = np.logspace(-2, 4, 13)
gammas = np.logspace(-4, 2, 13)

grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                    {"svc__C": Cs, "svc__gamma": gammas},
                    cv=SKF, scoring="accuracy", n_jobs=1).fit(Xm_tr, ym_tr)

scores = grid.cv_results_["mean_test_score"].reshape(len(Cs), len(gammas))
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(scores, cmap="viridis", aspect="auto", origin="lower")
ax.set_xticks(range(len(gammas))); ax.set_xticklabels([f"{g:.0e}" for g in gammas],
                                                     rotation=45, fontsize=7)
ax.set_yticks(range(len(Cs))); ax.set_yticklabels([f"{c:.0e}" for c in Cs], fontsize=7)
ax.set_xlabel("gamma"); ax.set_ylabel("C")
best_i = np.unravel_index(np.argmax(scores), scores.shape)
ax.scatter([best_i[1]], [best_i[0]], marker="*", s=300, color="crimson",
           edgecolor="white", label="best")
plt.colorbar(im, label="CV accuracy")
ax.set_title("The good region is a diagonal band, not a point")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"Best parameters : {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")
print(f"Test accuracy   : {grid.score(Xm_te, ym_te):.4f}")
print(f"Default settings: "
      f"{cross_val_score(make_pipeline(StandardScaler(), SVC()), Xm_tr, ym_tr, cv=SKF).mean():.4f}")

In [ ]:
# The diagonal band, explained
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
combos = [(0.1, 0.01, "low C, low gamma\n= underfit"),
          (1000, 100, "high C, high gamma\n= overfit"),
          (1000, 0.01, "high C, low gamma\n= nearly linear"),
          (grid.best_params_["svc__C"], grid.best_params_["svc__gamma"], "tuned")]
for ax, (C, g, label) in zip(axes, combos):
    mdl = SVC(kernel="rbf", C=C, gamma=g).fit(Xm_tr, ym_tr)
    boundary(mdl, Xm_tr, ym_tr, ax,
             f"{label}\ntrain {mdl.score(Xm_tr, ym_tr):.3f} / test {mdl.score(Xm_te, ym_te):.3f}")
plt.tight_layout(); plt.show()

print("Intuition for the diagonal: gamma controls how wiggly the boundary CAN be, and C")
print("controls how hard the model tries to use that wiggle. Raising one while lowering")
print("the other keeps the effective flexibility roughly constant.")

---
## 7.7 Multiclass SVMs, and probabilities

`SVC` handles multiclass with **one-vs-one**: it fits $\binom{K}{2}$ binary classifiers and
votes. `LinearSVC` uses **one-vs-rest** ($K$ classifiers). For a handful of classes either is
fine; one-vs-one costs more models but each is trained on less data.

**Probabilities are a genuine weakness.** An SVM outputs a *distance* from the boundary, not a
probability. `probability=True` fits **Platt scaling** — a logistic regression on the decision
values, cross-validated internally. It works, but:

- it makes fitting ~5× slower
- `predict_proba` and `predict` can occasionally disagree
- a better approach is usually `CalibratedClassifierCV` around a plain `SVC`

If you need well-calibrated probabilities, logistic regression is the more natural tool.

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=0,
                                              stratify=yw)

ovo = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10)).fit(Xw_tr, yw_tr)
ovr = make_pipeline(StandardScaler(), LinearSVC(C=1.0, max_iter=20000)).fit(Xw_tr, yw_tr)

print(f"one-vs-one SVC     : test accuracy {ovo.score(Xw_te, yw_te):.4f}")
print(f"one-vs-rest LinearSVC: test accuracy {ovr.score(Xw_te, yw_te):.4f}")
print(f"\nSVC fits {len(wine.target_names)*(len(wine.target_names)-1)//2} binary models "
      f"(one-vs-one); LinearSVC fits {len(wine.target_names)} (one-vs-rest).")
print(f"decision_function shape for SVC: {ovo.decision_function(Xw_te).shape}")
print()
print(classification_report(yw_te, ovo.predict(Xw_te), target_names=wine.target_names))

In [ ]:
# Calibration: raw decision values vs Platt scaling vs CalibratedClassifierCV
Xc2, yc2 = make_classification(n_samples=3_000, n_features=12, n_informative=6,
                               n_redundant=2, weights=[0.7, 0.3], random_state=1)
Xa2, Xb2, ya2, yb2 = train_test_split(Xc2, yc2, test_size=0.3, random_state=0, stratify=yc2)

plain_svc = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0)).fit(Xa2, ya2)
platt = make_pipeline(StandardScaler(),
                     SVC(kernel="rbf", C=1.0, probability=True, random_state=0)).fit(Xa2, ya2)
calib = make_pipeline(StandardScaler(),
                     CalibratedClassifierCV(SVC(kernel="rbf", C=1.0), method="isotonic",
                                            cv=5)).fit(Xa2, ya2)
logit2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xa2, ya2)

plt.plot([0, 1], [0, 1], "k--", lw=1.4, label="perfect calibration")
for name, mdl in [("SVC + Platt (probability=True)", platt),
                  ("SVC + isotonic calibration", calib),
                  ("logistic regression", logit2)]:
    pb = mdl.predict_proba(Xb2)[:, 1]
    fp, mp = calibration_curve(yb2, pb, n_bins=10, strategy="quantile")
    plt.plot(mp, fp, "o-", lw=2, label=f"{name} (log loss {log_loss(yb2, pb):.4f})")
plt.xlabel("mean predicted probability"); plt.ylabel("observed fraction positive")
plt.title("SVM probabilities need calibration; logistic regression does not")
plt.legend(fontsize=8); plt.show()

print(f"Plain SVC decision_function range: "
      f"[{plain_svc.decision_function(Xb2).min():.2f}, "
      f"{plain_svc.decision_function(Xb2).max():.2f}] -- not a probability")
print(f"ROC-AUC from raw decision values : "
      f"{roc_auc_score(yb2, plain_svc.decision_function(Xb2)):.4f}")
print(f"ROC-AUC from Platt probabilities : "
      f"{roc_auc_score(yb2, platt.predict_proba(Xb2)[:, 1]):.4f}")
print("\nNote the AUCs are nearly identical: calibration changes the numbers, not the")
print("ranking. Use decision_function for ranking and AUC; calibrate only if you need")
print("the probabilities themselves.")

In [ ]:
# The cost of probability=True
for name, est in [("SVC (no probabilities)", SVC(kernel="rbf")),
                  ("SVC (probability=True)", SVC(kernel="rbf", probability=True,
                                                 random_state=0))]:
    t0 = time.perf_counter()
    make_pipeline(StandardScaler(), est).fit(Xa2, ya2)
    print(f"  {name:<26} fit time {time.perf_counter()-t0:.3f}s")
print("\nPlatt scaling runs an internal 5-fold cross-validation, so it costs roughly 5x.")

---
## 7.8 Support Vector Regression (SVR)

The same idea, inverted. Instead of maximising a margin *between* classes, SVR fits a tube of
width $\epsilon$ around the regression function and **ignores every point inside it**:

$$\min\ \tfrac{1}{2}\|\mathbf{w}\|^2 + C\sum_i(\xi_i + \xi_i^*)
\quad\text{s.t.}\quad |y_i - f(\mathbf{x}_i)| \le \epsilon + \xi_i$$

This is **$\epsilon$-insensitive loss**: errors smaller than $\epsilon$ cost nothing. The
consequences:

- Only points **outside** the tube become support vectors → a sparse model
- **Robust to small noise**, because small residuals are free
- $\epsilon$ is a new hyperparameter: bigger tube, fewer support vectors, smoother fit

In [ ]:
xs = np.sort(rng.uniform(0, 10, 120))
ys = np.sin(xs) + 0.3*xs + rng.normal(0, 0.35, 120)
Xr = xs.reshape(-1, 1)
grid_x = np.linspace(-1, 12, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (kern, C, eps) in zip(axes, [("linear", 1.0, 0.5), ("rbf", 1.0, 0.5),
                                     ("rbf", 100.0, 0.1), ("rbf", 1.0, 1.5)]):
    mdl = make_pipeline(StandardScaler(), SVR(kernel=kern, C=C, epsilon=eps)).fit(Xr, ys)
    pred = mdl.predict(grid_x)
    ax.scatter(xs, ys, s=14, alpha=0.55, color="steelblue")
    ax.plot(grid_x, pred, color="crimson", lw=2)
    ax.fill_between(grid_x.ravel(), pred-eps, pred+eps, color="crimson", alpha=0.15)
    sv = mdl[-1].support_
    ax.scatter(xs[sv], ys[sv], s=70, facecolors="none", edgecolors="gold", linewidths=1.4)
    cvv = -cross_val_score(make_pipeline(StandardScaler(), SVR(kernel=kern, C=C, epsilon=eps)),
                           Xr, ys, cv=CV, scoring="neg_root_mean_squared_error").mean()
    ax.set_title(f"{kern}, C={C}, eps={eps}\n{len(sv)} SVs, CV RMSE {cvv:.3f}", fontsize=9)
    ax.set_ylim(-2.5, 5.5)
plt.tight_layout(); plt.show()

print("Shaded band = the epsilon tube. Gold circles = support vectors (points outside it).")
print("Bigger epsilon means a fatter tube, fewer support vectors, and a smoother fit.")
print("\nUnlike trees and KNN, SVR with a linear kernel CAN extrapolate. With an RBF kernel")
print("it reverts to the mean far from the data, because all kernel values decay to 0.")

In [ ]:
# SVR on real data, tuned
from sklearn.datasets import load_diabetes
dia = load_diabetes()
Xd, yd = dia.data, dia.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0)

svr_grid = GridSearchCV(make_pipeline(StandardScaler(), SVR(kernel="rbf")),
                        {"svr__C": np.logspace(0, 4, 9),
                         "svr__gamma": np.logspace(-4, 0, 9),
                         "svr__epsilon": [1, 5, 20]},
                        cv=CV, scoring="neg_root_mean_squared_error", n_jobs=1).fit(Xd_tr, yd_tr)

from sklearn.linear_model import RidgeCV
ridge = make_pipeline(StandardScaler(), RidgeCV()).fit(Xd_tr, yd_tr)
lin_svr = make_pipeline(StandardScaler(), SVR(kernel="linear", C=100)).fit(Xd_tr, yd_tr)

print(f"Best SVR parameters: {svr_grid.best_params_}\n")
print(f"{'model':<26}{'test RMSE':>11}{'test R2':>10}")
for name, mdl in [("Ridge", ridge), ("linear SVR", lin_svr), ("tuned RBF SVR", svr_grid)]:
    pr = mdl.predict(Xd_te)
    print(f"{name:<26}{np.sqrt(mean_squared_error(yd_te, pr)):>11.2f}"
          f"{r2_score(yd_te, pr):>10.4f}")
print("\nOn a small, noisy, roughly linear dataset the SVM's flexibility buys little.")
print("Note also how much tuning it needed (243 fits) to get there.")

---
## 7.9 Computational cost, and when to use an SVM

The kernel SVM solver is roughly $O(n^2)$ to $O(n^3)$ in the number of samples, and it must
store an $n \times n$ kernel matrix. That is the binding constraint.

| $n$ | Practical? |
|---|---|
| < 10,000 | Yes, comfortably |
| 10,000–100,000 | Slow; use `LinearSVC`/`LinearSVR`, or `SGDClassifier(loss="hinge")` |
| > 100,000 | Not with a kernel. Use a linear SVM, or approximate the kernel (`Nystroem`, `RBFSampler`) |

**Use an SVM when**

- $n$ is small-to-medium and the boundary is genuinely non-linear
- $p > n$ (text, genomics) — the margin formulation handles this gracefully with a linear kernel
- You want a sparse model defined by a few examples
- Accuracy matters more than interpretability or probability quality

**Avoid when**

- $n$ is large
- You need calibrated probabilities or feature importances
- Features are mixed-type or need heavy preprocessing (trees are easier)
- Someone has to explain individual decisions

In [ ]:
print(f"{'n_train':>9}{'kernel SVC (s)':>16}{'LinearSVC (s)':>15}{'logistic (s)':>14}")
for n_ in (500, 2_000, 8_000, 20_000):
    Xb_, yb_ = make_classification(n_samples=n_, n_features=20, n_informative=10,
                                   random_state=0)
    row = [n_]
    for est in [SVC(kernel="rbf"), LinearSVC(max_iter=5000, dual="auto"),
                LogisticRegression(max_iter=2000)]:
        t0 = time.perf_counter()
        make_pipeline(StandardScaler(), est).fit(Xb_, yb_)
        row.append(time.perf_counter() - t0)
    print(f"{row[0]:>9,}{row[1]:>16.2f}{row[2]:>15.2f}{row[3]:>14.2f}")
print("\nThe kernel SVM's time grows super-linearly; the linear models barely notice.")
print("Extrapolate the first column and you can see why kernel SVMs disappeared from")
print("large-scale problems.")

In [ ]:
# Kernel approximation: RBF-like power at linear cost
from sklearn.kernel_approximation import Nystroem, RBFSampler
from sklearn.linear_model import SGDClassifier

Xbig, ybig = make_classification(n_samples=20_000, n_features=20, n_informative=10,
                                 n_clusters_per_class=3, random_state=0)
Xa3, Xb3, ya3, yb3 = train_test_split(Xbig, ybig, test_size=0.3, random_state=0, stratify=ybig)

options = {
    "LinearSVC": make_pipeline(StandardScaler(), LinearSVC(max_iter=5000, dual="auto")),
    "Nystroem(300) + LinearSVC": make_pipeline(StandardScaler(),
                                               Nystroem(n_components=300, random_state=0),
                                               LinearSVC(max_iter=5000, dual="auto")),
    "RBFSampler(500) + SGD": make_pipeline(StandardScaler(),
                                           RBFSampler(n_components=500, random_state=0),
                                           SGDClassifier(loss="hinge", max_iter=200,
                                                         random_state=0)),
    "kernel SVC (rbf)": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}
for name, est in options.items():
    t0 = time.perf_counter()
    est.fit(Xa3, ya3)
    fit_s = time.perf_counter() - t0
    print(f"  {name:<28} test accuracy {est.score(Xb3, yb3):.4f}   fit {fit_s:>6.2f}s")
print("\nKernel approximation gets most of the non-linear accuracy at a fraction of the")
print("cost. This is the standard trick when n is too large for an exact kernel SVM.")

---
## Exercises

**Exercise 1.** On the breast cancer dataset, compare a linear SVM, an RBF SVM (tuned) and
logistic regression. Report CV accuracy, test ROC-AUC, the number of support vectors, and fit
time. Which would you deploy?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
Xbc, ybc = bc.data, bc.target
Xa4, Xb4, ya4, yb4 = train_test_split(Xbc, ybc, test_size=0.25, random_state=0, stratify=ybc)

rbf_grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                        {"svc__C": np.logspace(-1, 3, 9),
                         "svc__gamma": np.logspace(-4, 0, 9)},
                        cv=SKF, scoring="roc_auc", n_jobs=1).fit(Xa4, ya4)
print(f"Best RBF parameters: {rbf_grid.best_params_}\n")

print(f"{'model':<26}{'CV AUC':>9}{'test AUC':>10}{'test acc':>10}{'SVs':>7}{'fit s':>8}")
for name, est in [("logistic regression",
                   make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
                  ("linear SVM (C=1)", make_pipeline(StandardScaler(), SVC(kernel="linear"))),
                  ("RBF SVM (tuned)", rbf_grid.best_estimator_)]:
    t0 = time.perf_counter()
    cvv = cross_val_score(est, Xa4, ya4, cv=SKF, scoring="roc_auc").mean()
    fit_s = (time.perf_counter() - t0) / 5
    est.fit(Xa4, ya4)
    scores = est.decision_function(Xb4) if hasattr(est, "decision_function") else est.predict_proba(Xb4)[:, 1]
    n_sv = len(est[-1].support_) if isinstance(est[-1], SVC) else np.nan
    print(f"{name:<26}{cvv:>9.4f}{roc_auc_score(yb4, scores):>10.4f}"
          f"{est.score(Xb4, yb4):>10.4f}{n_sv:>7}{fit_s:>8.3f}")

print("\nWhich to deploy: logistic regression, unless the AUC gap is material for the")
print("clinical decision. Reasons: it is as accurate here, it produces calibrated")
print("probabilities that a clinician can act on with a chosen threshold, it exposes")
print("coefficients (odds ratios per measurement) that a pathologist can sanity-check,")
print("and it is 30 features x 1 dot product at inference instead of a kernel evaluation")
print("against dozens of support vectors.")
print("\nThe SVM would be the right pick if the boundary were strongly non-linear -- and")
print("the near-identical scores are the evidence that it is not.")

**Exercise 2.** Demonstrate the `C`-`gamma` interaction. Fit RBF SVMs across a grid on the
moons data and report, for each combination, the training accuracy, test accuracy and number of
support vectors. Identify the overfitting and underfitting corners.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
Cs2 = [0.01, 1, 100, 10_000]
gs2 = [0.001, 0.1, 10, 1000]

rows = []
for C in Cs2:
    for g in gs2:
        mdl = SVC(kernel="rbf", C=C, gamma=g).fit(Xm_tr, ym_tr)
        rows.append({"C": C, "gamma": g,
                     "train_acc": mdl.score(Xm_tr, ym_tr),
                     "test_acc": mdl.score(Xm_te, ym_te),
                     "gap": mdl.score(Xm_tr, ym_tr) - mdl.score(Xm_te, ym_te),
                     "n_SV": len(mdl.support_),
                     "SV_pct": len(mdl.support_) / len(ym_tr)})
g2 = pd.DataFrame(rows)
print(g2.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a_, col, title in zip(ax, ["train_acc", "test_acc", "SV_pct"],
                          ["Training accuracy", "Test accuracy",
                           "Fraction of points that are support vectors"]):
    pivot = g2.pivot(index="C", columns="gamma", values=col)
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", ax=a_, cbar=False)
    a_.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

worst_over = g2.loc[g2.gap.idxmax()]
worst_under = g2.loc[g2.test_acc.idxmin()]
print(f"\nOVERFITTING corner : C={worst_over.C}, gamma={worst_over.gamma} -- "
      f"train {worst_over.train_acc:.3f}, test {worst_over.test_acc:.3f}, "
      f"gap {worst_over.gap:.3f}, {worst_over.SV_pct:.0%} of points are SVs")
print(f"UNDERFITTING corner: C={worst_under.C}, gamma={worst_under.gamma} -- "
      f"train {worst_under.train_acc:.3f}, test {worst_under.test_acc:.3f}")
print("\nDiagnostic rule of thumb: if most of your training points are support vectors,")
print("the model is either badly under-regularised (high C, high gamma) or the features")
print("are unscaled. Both are worth checking before you tune anything else.")

**Exercise 3.** SVMs are strong when $p > n$. Build a dataset with 200 features and 60 samples
(only 10 features informative) and compare a linear SVM, an RBF SVM, logistic regression with
L2, and a random forest.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

Xw2, yw2 = make_classification(n_samples=60, n_features=200, n_informative=10,
                               n_redundant=5, n_classes=2, random_state=3)
print(f"n = {Xw2.shape[0]}, p = {Xw2.shape[1]}  ->  p is more than 3x n\n")

candidates = {
    "linear SVM (C=1)": make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)),
    "linear SVM (C=0.01)": make_pipeline(StandardScaler(), SVC(kernel="linear", C=0.01)),
    "RBF SVM (defaults)": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    "logistic regression (L2)": make_pipeline(StandardScaler(),
                                              LogisticRegression(max_iter=5000)),
    "logistic regression (L1)": make_pipeline(StandardScaler(),
                                              LogisticRegression(penalty="l1",
                                                                 solver="liblinear", C=0.1)),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=0),
}
print(f"{'model':<28}{'CV accuracy':>13}{'CV sd':>8}")
res = []
for name, est in candidates.items():
    cvv = cross_val_score(est, Xw2, yw2, cv=SKF, scoring="accuracy")
    res.append((name, cvv.mean(), cvv.std()))
    print(f"{name:<28}{cvv.mean():>13.4f}{cvv.std():>8.4f}")

print(f"\nBaseline (majority class): "
      f"{cross_val_score(DummyClassifier(strategy='most_frequent'), Xw2, yw2, cv=SKF).mean():.4f}")
print()
print("Why the linear SVM does well here:")
print("  * with p > n the data is ALWAYS linearly separable, so the question is not")
print("    'can we separate it' but 'which separator generalises' -- exactly what the")
print("    maximum-margin criterion answers")
print("  * the margin formulation regularises implicitly; note that lowering C (more")
print("    regularisation) helps")
print("  * the RBF kernel adds flexibility this problem does not need, and with 60 points")
print("    that flexibility is pure variance")
print("  * the forest has too little data per tree to find the 10 informative columns")
print("    among 200")
print("\nThis is the regime where SVMs earned their reputation: text classification and")
print("genomics, both of which have far more features than samples.")

**Exercise 4 (challenge).** You must classify 200,000 transactions in real time, with a 5 ms
budget per prediction. An RBF SVM is the most accurate model you have found. Work out whether
you can ship it, and design the best alternative if you cannot.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
Xl, yl = make_classification(n_samples=40_000, n_features=25, n_informative=12,
                             n_clusters_per_class=4, class_sep=0.9, random_state=0)
Xa5, Xb5, ya5, yb5 = train_test_split(Xl, yl, test_size=0.25, random_state=0, stratify=yl)
print(f"Using {len(ya5):,} training rows as a stand-in for 200,000 (an exact kernel SVM")
print(f"on the full set would need an {200_000**2/1e9:.0f}-billion-entry kernel matrix).\n")

print("STEP 1 -- measure the exact kernel SVM's cost and accuracy")
t0 = time.perf_counter()
svc_big = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, cache_size=500)).fit(Xa5, ya5)
fit_s = time.perf_counter() - t0
t0 = time.perf_counter()
svc_big.predict(Xb5[:1000])
per_pred_ms = (time.perf_counter() - t0) / 1000 * 1000

print(f"  fit time on {len(ya5):,} rows : {fit_s:.1f}s")
print(f"  support vectors             : {len(svc_big[-1].support_):,} "
      f"({len(svc_big[-1].support_)/len(ya5):.1%} of the training set)")
print(f"  latency per prediction      : {per_pred_ms:.3f} ms")
print(f"  test accuracy               : {svc_big.score(Xb5, yb5):.4f}")
print(f"\n  A single prediction must evaluate the kernel against every support vector,")
print(f"  so latency scales with the number of SVs -- and the SV count grows with n.")

In [ ]:
print("STEP 2 -- extrapolate to 200,000 rows")
ns = [2_000, 5_000, 10_000, 20_000, 40_000]
fits, svs, lats = [], [], []
for n_ in ns:
    idx = rng.choice(len(ya5), n_, replace=False)
    t0 = time.perf_counter()
    m_ = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, cache_size=500)).fit(
        Xa5[idx], ya5[idx])
    fits.append(time.perf_counter() - t0)
    svs.append(len(m_[-1].support_))
    t0 = time.perf_counter(); m_.predict(Xb5[:500]); lats.append((time.perf_counter()-t0)/500*1000)

scaling = pd.DataFrame({"n_train": ns, "fit_s": np.round(fits, 2),
                        "support_vectors": svs, "latency_ms": np.round(lats, 3)})
print(scaling.to_string(index=False))

# fit a power law to the latency growth
coef = np.polyfit(np.log(ns), np.log(lats), 1)
proj = np.exp(coef[1]) * 200_000 ** coef[0]
print(f"\n  Latency grows roughly as n^{coef[0]:.2f}.")
print(f"  Projected latency at n = 200,000: {proj:.2f} ms  vs a 5 ms budget")
print(f"  VERDICT: {'FITS' if proj < 5 else 'DOES NOT FIT'} the latency budget.")
print(f"  Projected fit time also grows super-linearly, making retraining impractical.")

In [ ]:
print("STEP 3 -- design the alternative\n")
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

alternatives = {
    "logistic regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=3000)),
    "LinearSVC": make_pipeline(StandardScaler(), LinearSVC(max_iter=5000, dual="auto")),
    "Nystroem(500) + LinearSVC": make_pipeline(StandardScaler(),
                                               Nystroem(n_components=500, gamma=0.05,
                                                        random_state=0),
                                               LinearSVC(max_iter=5000, dual="auto")),
    "Nystroem(1000) + SGD hinge": make_pipeline(StandardScaler(),
                                                Nystroem(n_components=1000, gamma=0.05,
                                                         random_state=0),
                                                SGDClassifier(loss="hinge", max_iter=300,
                                                              random_state=0)),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
print(f"{'model':<30}{'accuracy':>10}{'fit s':>9}{'latency ms':>13}")
print(f"{'exact RBF SVM (reference)':<30}{svc_big.score(Xb5, yb5):>10.4f}"
      f"{fit_s:>9.1f}{per_pred_ms:>13.3f}")
for name, est in alternatives.items():
    t0 = time.perf_counter(); est.fit(Xa5, ya5); f_s = time.perf_counter() - t0
    t0 = time.perf_counter(); est.predict(Xb5[:2000]); lat = (time.perf_counter()-t0)/2000*1000
    print(f"{name:<30}{est.score(Xb5, yb5):>10.4f}{f_s:>9.1f}{lat:>13.4f}")

In [ ]:
print("\n" + "="*72)
print("RECOMMENDATION")
print("="*72)
print("Do NOT ship the exact RBF SVM. Two independent blockers:")
print("  1. Latency scales with the number of support vectors, which scales with n.")
print("     The projection above exceeds the 5 ms budget before we even add feature")
print("     computation and network overhead.")
print("  2. Training becomes impractical: super-linear fit time plus an O(n^2) kernel")
print("     matrix means retraining on fresh fraud patterns would take hours to days.")
print()
print("Ship instead: Nystroem kernel approximation feeding a linear model.")
print("  * it approximates the same RBF feature space with a fixed number of components,")
print("    so inference is one matrix multiply plus one dot product -- CONSTANT in n")
print("  * accuracy lands close to the exact kernel (see the table above)")
print("  * training is linear in n, so nightly retraining is trivial")
print("  * n_components is a direct accuracy/latency dial you can tune to the budget")
print()
print("Also worth benchmarking: histogram gradient boosting. It is usually as accurate on")
print("tabular data, has fast batched inference, handles missing values natively, and")
print("gives feature importances -- which a fraud analyst will ask for.")
print()
print("Engineering notes for whichever wins:")
print("  * measure p99 latency, not the mean; a 5 ms budget is a tail requirement")
print("  * quantise or compile the model (ONNX) if you need the last few milliseconds")
print("  * cache feature computation -- it is often the real bottleneck, not the model")
print("  * keep the exact SVM as an offline benchmark, so you know what accuracy the")
print("    latency constraint is costing you")
print("="*72)

---
## Summary

| Concept | Key point |
|---|---|
| Maximum margin | Widest corridor between classes; $2/\|\mathbf{w}\|$ |
| Support vectors | The only points that define the model |
| Soft margin | Allows violations at cost `C` |
| `C` | **Inverse** regularisation: small = wide margin, more bias |
| Hinge loss | Zero once a point is correct with margin ≥ 1 (unlike log loss) |
| Kernel trick | Inner products in a high-dimensional space, computed cheaply |
| RBF kernel | The default; `gamma` sets the influence radius |
| `gamma` | Small = smooth/high bias; large = wiggly/high variance |
| Scaling | **Mandatory** — the kernel and the margin both depend on it |
| Tuning | `C` and `gamma` interact; grid over powers of 10 |
| Multiclass | One-vs-one (`SVC`) or one-vs-rest (`LinearSVC`) |
| Probabilities | Not native; Platt scaling costs ~5× and calibration is imperfect |
| SVR | $\epsilon$-insensitive tube; points inside cost nothing |
| Cost | $O(n^2)$–$O(n^3)$; use linear SVMs or kernel approximation above ~50k rows |
| Best use | Small-to-medium $n$, non-linear boundary, or $p > n$ |

**Next up:** [Notebook 8 — Naive Bayes](8.%20Naive%20Bayes.ipynb), a model built directly on
Bayes' theorem, with an assumption so wrong it should not work — and yet it does.